# pdfspine 试验：原始 PDF → Markdown（pdfspine）→ 建库 → 与文字层对比

用 **pdfspine**（纯 Rust 的 PDF 库，Apache-2.0；**完全离线**：不联网、不需要 GPU / 模型下载、不调用任何外部服务）代替默认的 PyPDF2 文字层，
把 `PDF_DIR` 下（递归，含各子文件夹）的 PDF 转成 Markdown 并建库。产物格式与 `batch_qa.ipynb` 的步骤 1-2 完全兼容，但放在**独立的工作目录**里，不碰文字层的 `md/`、`store/`，便于对比：

1. **步骤 1：PDF → Markdown**（`superindex.extractors.pdfspine.convert_pdf`）：每页前 `<!-- page: N -->`（空页也保留标记，页码与 PDF 一致），
   表格转成 Markdown 表格，按字号出 `#` 标题（不再用 flash 补标题）。`meta.json` 字段同 batch_qa，`extractor` 为 `"pdfspine"`；
   无文字层 / 密码不对的 PDF 跳过并登记到 `skipped_pdfs.json`，规则同 batch_qa。
2. **步骤 2：建库**：与 batch_qa 步骤 2 同一段代码（`index_markdown`、文档元数据 / 报告类型 / 报告期、不同文件夹同名 PDF 加前缀）。
3. **步骤 3：对比统计**（只有数字，不输出任何内容）：按文件夹统计 PDF 数、成功 / 跳过 / 失败数、表格数与表格行数、标题数、空页数、平均每页字符数；
   batch_qa 的文字层 `md/` 存在时并列给出同类统计。另存 `compare.csv`。
4. **用这份库跑问答**：不改 batch_qa 的代码，只改配置，见文末。

| | Databricks | 本地（仓库内） |
|---|---|---|
| 安装 | 运行「环境安装」cell（`%pip install ... pdfspine==0.12.0` + 重启 Python） | `uv sync --group notebook --extra pdfspine`；**不要运行**「环境安装」cell |
| 模型 | 只有文档元数据（`DOC_META=True`，默认）会调模型，配置同 batch_qa 的「模型配置」cell；没配模型时只用文件名规则 | `.env`，同 batch_qa |
| 数据 | `PDF_DIR` 指向 `/Volumes/...` 之类的路径 | 本地路径 |

**配置与 batch_qa 共用**：`DATASET_PATH`、`PDF_DIR`、`PDF_INGEST_PASSWORD` / `PDF_PASSWORDS`、`NB_WORK_DIR`、`PERSIST_DIR`、`DOC_META` 等同名配置（环境变量 / `.env`）含义相同，同一份 `.env` 两个 Notebook 都能用。
本 Notebook 自己的只有两项：

- `PDFSPINE_WORK_DIR`：pdfspine 的工作目录，默认是 batch_qa 工作目录名后加 `_pdfspine`（如 `<ROOT_DIR>/data/notebook/<题集名>_pdfspine/`）；
- `PDFSPINE_PERSIST_DIR`：持久目录，默认是 batch_qa 的 `PERSIST_DIR` 名后加 `_pdfspine`（没设 `PERSIST_DIR` 则不同步）。

**带密码的 PDF**：候选密码同 batch_qa（`PDF_INGEST_PASSWORD` 主密码排第一，再按 `PDF_PASSWORDS` 顺序）。pdfspine 自己支持 RC4 / AES-128 / AES-256 解密，
在内存里用密码打开，**不生成解密副本**、不需要 `pycryptodome`；`meta.json` 只记 `encrypted` / `password_used`，不记密码。**Markdown 是明文**，工作目录 / 持久目录的访问权限要按原 PDF 的密级管理。

**产出物**（工作目录下，与 batch_qa 同样的布局）

| 路径 | 内容 |
|---|---|
| `md/` | PDF 转出的 Markdown 及同名 `*.meta.json`（目录结构同 `PDF_DIR`） |
| `skipped_pdfs.json` | 被跳过的 PDF（无文字层 / 密码不正确或未配置）及原因 |
| `store/` | 索引库 |
| `compare.csv` | 步骤 3 的对比统计（utf-8-sig） |

**命令行执行（本地）**：必须带上跳过「环境安装」cell 的参数：

```bash
uv run --group notebook --extra pdfspine jupyter nbconvert --to notebook --execute notebooks/pdfspine_ingest.ipynb \
  --TagRemovePreprocessor.enabled=True --TagRemovePreprocessor.remove_cell_tags databricks-setup \
  --output-dir <某目录>
```

## 环境安装（仅 Databricks）

**只在 Databricks 上运行下面两个 cell**（cell tag `databricks-setup`），本地不要运行。第一个 cell 只放 `%pip install`：默认从 PyPI 装 `superindex==0.1.2` 和固定版本的 `pdfspine==0.12.0`
（pdfspine 会一并装上 `ocrspine-models`，是本试验用不到的 OCR 模型数据包，只占磁盘、不联网）；集群访问不了公网 PyPI 时换成内部镜像写法，如
`%pip install --index-url https://<公司内部 PyPI 镜像>/simple superindex==0.1.2 pdfspine==0.12.0 pandas`。第二个 cell 重启 Python，让新装的包生效。

In [ ]:
%pip install superindex==0.1.2 pdfspine==0.12.0 pandas

In [ ]:
import os

# 只在 Databricks 上重启 Python（让上一格 %pip 装的包生效）；其它环境什么都不做
if os.environ.get("DATABRICKS_RUNTIME_VERSION") and "dbutils" in globals():
    dbutils.library.restartPython()  # noqa: F821 - dbutils 由 Databricks 注入
else:
    print("非 Databricks 环境，跳过")

In [ ]:
# ───────────── 模型配置（必须在任何 import superindex / litellm 之前运行）─────────────
# 本 Notebook 只有建库时的文档元数据（DOC_META=True）、摘要（INDEX_SUMMARY=True）会调模型，写法同 batch_qa。
# 本 cell 只写环境变量，不 import litellm / superindex，所以下面这些变量能赶在它们之前生效。
# 默认不改动任何环境变量：本地有 .env 时行为不变（.env 在下一格"环境"里才加载，且不覆盖这里已设的变量）。
import os

os.environ.setdefault("LITELLM_LOCAL_MODEL_COST_MAP", "True")  # litellm 用内置价格表，import 时不联网拉取
os.environ.setdefault("OPENAI_AGENTS_DISABLE_TRACING", "1")    # 关闭 openai-agents 的 tracing 上报

# 想改哪项就取消注释并填值；值为空串的项会被跳过。**不要把密钥明文写进 Notebook**。
# 在 Databricks 上从 secrets 读：dbutils.secrets.get(scope="<scope>", key="<key>")
LLM_CONFIG: dict[str, str] = {
    # ── 方式一：OpenAI 兼容端点（vLLM、公司网关等）：端点上的模型名（自动加 openai/ 前缀），地址一般带 /v1 ──
    # 只有 OPENAI_MODEL 非空才启用这一组；对应的 SUPERINDEX_* 已设（非空）时以 SUPERINDEX_* 为准。映射在下一格"环境"里做。
    # "OPENAI_MODEL": "<模型名>",
    # "OPENAI_BASE_URL": "https://<公司网关>/v1",
    # "OPENAI_API_KEY": dbutils.secrets.get(scope="<scope>", key="<key>"),
    # 等价的 SUPERINDEX_* 写法（也可以用）：
    # "SUPERINDEX_CHAT_MODEL": "openai/<模型名>",
    # "SUPERINDEX_BASE_URL": "https://<公司网关>/v1",
    # "SUPERINDEX_API_KEY_OVERRIDE": dbutils.secrets.get(scope="<scope>", key="<key>"),
    #
    # ── 方式二：Databricks Model Serving：databricks/<serving endpoint 名> ──
    # "SUPERINDEX_CHAT_MODEL": "databricks/<endpoint 名>",
    # "SUPERINDEX_BASE_URL": "https://<workspace 域名>/serving-endpoints",
    # "SUPERINDEX_API_KEY_OVERRIDE": dbutils.secrets.get(scope="<scope>", key="<key>"),  # Databricks token
    #
    # ── 方式三：Azure OpenAI：azure/<部署名> ──
    # "SUPERINDEX_CHAT_MODEL": "azure/<deployment 名>",
    # "AZURE_API_BASE": "https://<资源名>.openai.azure.com",
    # "AZURE_API_KEY": dbutils.secrets.get(scope="<scope>", key="<key>"),
    # "AZURE_API_VERSION": "<api 版本>",
    #
    # ── 方式四：公司自封装的 Python SDK：模型名写 company/<任意名>，并启用下面的"公司 SDK 适配器"cell ──
    # "SUPERINDEX_CHAT_MODEL": "company/<任意名>",
    #
    # ── 其它 ──
    # "SUPERINDEX_INDEX_MODEL": "",         # 只在 INDEX_SUMMARY=True 时用到；写法同 SUPERINDEX_CHAT_MODEL；用 OPENAI_MODEL 时不设则同用它
    # "SUPERINDEX_REASONING_EFFORT": "",    # 公司模型一般留空；设了而模型不支持会报 UnsupportedParamsError
}

for _name, _value in LLM_CONFIG.items():
    if _value:
        os.environ[_name] = _value
print("模型配置：写入环境变量 " + (", ".join(k for k, v in LLM_CONFIG.items() if v) or "（无，沿用 .env / 进程环境变量）"))

In [ ]:
# 确定唯一的绝对路径基准 ROOT_DIR（.env、data/、相对路径配置都以它为基准，不依赖 cwd、不 chdir）。必须在 import superindex 之前做。
# ROOT_DIR：环境变量 ROOT_DIR > 从 cwd 向上找到的仓库根。本 Notebook 必须在仓库里运行（Databricks 上先把仓库拉到最新再打开）。
import json
import os
import shutil
import sys
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path
from typing import Any

from dotenv import dotenv_values, load_dotenv
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path | None:
    """向上找含 pyproject.toml 和 superindex/ 包目录的目录（即本仓库根）；找不到（pip 安装、不在仓库里）返回 None。"""
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").is_file() and (folder / "superindex").is_dir():
            return folder
    return None


IS_DATABRICKS = bool(os.environ.get("DATABRICKS_RUNTIME_VERSION"))
_repo_root = find_repo_root(Path.cwd().resolve())
if _repo_root is None:
    raise RuntimeError(
        f"从当前目录 {Path.cwd().resolve()} 向上找不到 SuperIndex 仓库根（含 pyproject.toml 和 superindex/）。"
        "请在仓库里打开并运行本 Notebook（Databricks 上先把仓库拉到最新）。"
    )
ROOT_DIR = Path(os.environ.get("ROOT_DIR", "").strip() or _repo_root).expanduser().resolve()
sys.path.insert(0, str(_repo_root))  # import superindex 用仓库里的源码，而不是 %pip 装的 PyPI 版；不 chdir


def under_root(p: str | Path) -> Path:
    """绝对路径原样（resolve）；相对路径以 ROOT_DIR 为基准。"""
    p = Path(p).expanduser()
    return (p if p.is_absolute() else ROOT_DIR / p).resolve()


_root_from = "环境变量 ROOT_DIR" if os.environ.get("ROOT_DIR", "").strip() else "从当前目录向上找到的仓库根"
print(f"当前目录 : {Path.cwd().resolve()}")
print(f"ROOT_DIR : {ROOT_DIR}（来源：{_root_from}）")

from superindex.runtime import (
    LLMSettings,
    configure_litellm,
    getenv,
    set_offline_defaults,
)

set_offline_defaults()  # 必须早于任何 litellm 的 import
# .env 候选位置：先 ROOT_DIR/.env，再当前目录的 .env（与前者不是同一个文件时）。都 override=False，
# 所以优先级：进程环境变量 > ROOT_DIR/.env > 当前目录/.env。
_env_candidates = [ROOT_DIR / ".env"]
if (Path.cwd().resolve() / ".env").resolve() != _env_candidates[0].resolve():
    _env_candidates.append(Path.cwd().resolve() / ".env")
ENV_SOURCE: dict[str, str] = {k: "环境变量" for k in os.environ}  # 变量名 -> 来源（只记名字，不记值）
ENV_FILES: list[Path] = []
_dotenv_keys: set[str] = set()
for _env_file in _env_candidates:
    if not _env_file.is_file():
        print(f".env     : {_env_file} 不存在")
        continue
    _keys = sorted(k for k, v in dotenv_values(_env_file, encoding="utf-8").items() if v is not None)
    _labels = []
    for _k in _keys:
        _shadow = ENV_SOURCE.get(_k)  # 已有来源 = 环境变量，或前一个 .env
        ENV_SOURCE.setdefault(_k, f".env（{_env_file}）")
        _labels.append(_k if _shadow is None else _k + ("（已被环境变量覆盖）" if _shadow == "环境变量" else "（已被前一个 .env 覆盖）"))
    _dotenv_keys.update(_keys)
    ENV_FILES.append(_env_file)
    load_dotenv(_env_file, override=False, encoding="utf-8")  # 已设置的环境变量优先，不覆盖
    print(f".env     : {_env_file} 已加载，{len(_keys)} 个变量：{', '.join(_labels) or '（空）'}")  # 只打印变量名，绝不打印值
if not ENV_FILES:
    print(f"未读到任何 .env：配置项将使用环境变量或默认值。请把 .env 放到 {ROOT_DIR} 或 {Path.cwd().resolve()}")
_old_names = sorted(k for k in {*os.environ, *_dotenv_keys} if k.startswith("SI_NB_"))
if _old_names:
    _renamed = {"SI_NB_PDF_DIR": "PDF_DIR", "SI_NB_WORK_DIR": "NB_WORK_DIR", "SI_NB_DATASET": "DATASET_PATH",
                "SI_NB_RESULTS": "RESULTS_PATH", "SI_NB_EVAL_INPUT": "RESULTS_PATH"}
    print("⚠ 发现已废弃的变量名（不再生效）："
          + "；".join(f"{k} → " + ("请改为 " if i == 0 else "") + _renamed.get(k, k[len("SI_NB_"):])
                      for i, k in enumerate(_old_names)))


def apply_openai_env() -> dict[str, str]:
    """OPENAI_MODEL / OPENAI_BASE_URL / OPENAI_API_KEY -> SUPERINDEX_*（与新版 superindex.runtime 同一规则；
    这里自己做一遍，PyPI 旧版没有该回退时行为也一致）。只有 OPENAI_MODEL 非空才启用；
    SUPERINDEX_*（含旧名 PAGEINDEX_*）已设为非空的不覆盖。返回实际写入的 {变量名: 值}。"""
    model = os.environ.get("OPENAI_MODEL", "").strip()
    if not model:
        return {}
    model = model if model.startswith("openai/") else f"openai/{model}"  # 模型名含 "/" 也要加，别让 litellm 把前半当 provider
    wanted = {
        "SUPERINDEX_CHAT_MODEL": model,
        "SUPERINDEX_INDEX_MODEL": model,  # 建库摘要（INDEX_SUMMARY=True）同用这个模型
        "SUPERINDEX_BASE_URL": os.environ.get("OPENAI_BASE_URL", "").strip(),
        "SUPERINDEX_API_KEY_OVERRIDE": os.environ.get("OPENAI_API_KEY", "").strip(),
    }
    applied = {}
    for name, value in wanted.items():
        if value and not (getenv(name) or "").strip():
            os.environ[name] = applied[name] = value
    return applied


_applied = apply_openai_env()
if os.environ.get("OPENAI_MODEL", "").strip():
    print("模型配置：使用 OPENAI_MODEL / OPENAI_BASE_URL / OPENAI_API_KEY"
          f"（模型 {getenv('SUPERINDEX_CHAT_MODEL')}，地址 {getenv('SUPERINDEX_BASE_URL') or '（未设）'}；"
          f"已填入 {', '.join(_applied) or '（无，SUPERINDEX_* 均已设置）'}）")  # 不打印 key

from superindex import page_images
from superindex.engine.local_store import DocStore
from superindex.extractors.pdfspine import EXTRACTOR, convert_pdf, markdown_stats, require_pdfspine
from superindex.md_ingest import (RateLimitPolicy, backfill_doc_meta, clash_names, doc_meta_stats, drop_clash_leftovers,
                                  index_markdown, pdf_source_path, same_name_sources)

print(f"运行环境: {'Databricks' if IS_DATABRICKS else '本地'}")
print(f"仓库根目录: {_repo_root}")


def _superindex_version(src: Path) -> str:
    try:
        if (src.parent / "pyproject.toml").is_file():  # 源码树：读 pyproject 里的版本（已安装包的元数据可能是别的副本的）
            import tomllib

            return tomllib.loads((src.parent / "pyproject.toml").read_text(encoding="utf-8"))["project"]["version"]
        from importlib.metadata import version

        return version("superindex")
    except Exception:  # noqa: BLE001
        return "未知"


import superindex

_SRC = Path(superindex.__file__).resolve().parent
print(f"superindex 来自: {_SRC}（版本 {_superindex_version(_SRC)}）")
pdfspine = require_pdfspine()  # 没装 pdfspine 时在这里给出安装提示
print(f"pdfspine 版本: {pdfspine.__version__}（离线，不联网）")

In [ ]:
# ───────────── 配置（所有可调项都在这一格）─────────────
# 每项都可用同名环境变量覆盖（也可写进 ROOT_DIR/.env）。与 batch_qa 同名的配置项含义相同，同一份 .env 两个 Notebook 都能用。
def _env(name: str, default: Any, cast=str) -> Any:
    raw = os.environ.get(name)
    return default if raw is None or raw.strip() == "" else cast(raw.strip())


def _source(name: str) -> str:
    """配置项的值来自哪里：环境变量非空 → 查 ENV_SOURCE（环境变量 / .env（路径）），否则默认值。"""
    return ENV_SOURCE.get(name, "环境变量") if os.environ.get(name, "").strip() else "默认值"


def _raw_note(name: str, default: str) -> str:
    """原始写法是相对路径时，附带显示原始值（相对路径以 ROOT_DIR 为基准）。"""
    raw = os.environ.get(name, "").strip() or default
    return "" if Path(raw).expanduser().is_absolute() else f"（原始值 {raw}，以 ROOT_DIR 为基准）"


def _bool(text: str) -> bool:
    return text.lower() in ("1", "true", "yes", "on")


# --- 输入（同 batch_qa）---
# 问题文件本 Notebook 不读，只用它的文件名定默认工作目录（与 batch_qa 一致：data/notebook/<题集名>）
DATASET_PATH = under_root(_env("DATASET_PATH", "data/questions.jsonl"))
# 如 "/Volumes/<catalog>/<schema>/<volume>/qa/pdfs"、"/dbfs/<目录>/pdfs"
PDF_DIR = under_root(_env("PDF_DIR", "data/pdfs"))                          # 原始 PDF 目录（递归查找 *.pdf）
_pdf_state = (f"存在，递归找到 {sum(1 for p in PDF_DIR.rglob('*') if p.is_file() and p.suffix.lower() == '.pdf')} 个 *.pdf"
              if PDF_DIR.is_dir() else "不存在")
print(f"PDF 目录 : {PDF_DIR}   [来源：{_source('PDF_DIR')}]   [{_pdf_state}]" + _raw_note("PDF_DIR", "data/pdfs"))


# --- PDF → Markdown ---
PDF_EXTRACTOR = EXTRACTOR                            # 固定 "pdfspine"（写进 meta.json 的 extractor）
FORCE_EXTRACT = _env("FORCE_EXTRACT", False, _bool)  # True：已有 .md 也重新转换

# --- 带密码的 PDF（需要密码才能打开的 PDF；默认不处理）---
# 主密码放在环境变量 PDF_INGEST_PASSWORD 里（写进 .env 即可：PDF_INGEST_PASSWORD='你的密码'；上一格已加载 .env）。
# 它原样当作一个密码（不做 JSON 解析），总是排在候选列表第一个。.env 里密码含 #、空格、引号等字符时用单引号括起来。
# 其它候选密码填 PDF_PASSWORDS，排在主密码之后依次尝试。都没配 = 需要密码的 PDF 一律跳过并标记（不影响其它 PDF）。
# 不加密、或只设了 owner 密码（打开密码为空）的 PDF 不受影响，照常处理。
# 不要把明文密码写进 Notebook。Databricks 上没有 .env 时从 secrets 读（把下一行 [] 换成）：
#   PDF_PASSWORDS = [dbutils.secrets.get(scope="<scope>", key="<key>")]  # 多个密码就写多项
# 也可用环境变量 PDF_PASSWORDS 覆盖：值是 JSON 数组字符串，如 '["pw1", "pw,with,comma"]'（密码里可含逗号等任意字符，
# 与 FIELD_MAP 的 JSON 传法一致）；值不是 JSON 数组（不是合法 JSON，或是 JSON 数字/字符串）时，整个字符串当作单个密码。
# 密码不会被写进 meta.json / skipped_pdfs.json / 任何输出；步骤 1 只在内存里解密，不落解密副本。
PDF_PASSWORDS: list[str] = []


def parse_passwords(raw: str) -> list[str]:
    """环境变量值 -> 候选密码列表：JSON 数组按元素；其它（含非法 JSON）整个字符串当单个密码。"""
    try:
        data = json.loads(raw)
    except ValueError:
        return [raw]
    return [str(p) for p in data if p not in (None, "")] if isinstance(data, list) else [raw]


_env_passwords = os.environ.get("PDF_PASSWORDS")
if _env_passwords is not None and _env_passwords.strip():
    PDF_PASSWORDS = parse_passwords(_env_passwords)
_main_password = os.environ.get("PDF_INGEST_PASSWORD", "")
PDF_PASSWORDS = [_main_password, *PDF_PASSWORDS]  # 主密码排第一
PDF_PASSWORDS = list(dict.fromkeys(p for p in PDF_PASSWORDS if p))  # 去重保序；空串不是密码（空密码本来就会先试）

# --- 工作目录 ---
# batch_qa 的工作目录（文字层产物，规则同 batch_qa：NB_WORK_DIR > data/notebook/<题集名>）：本 Notebook 只读它的 md/ 做对比，绝不写入
TEXT_WORK_DIR = under_root(os.environ.get("NB_WORK_DIR", "").strip() or ROOT_DIR / "data" / "notebook" / DATASET_PATH.stem)
# pdfspine 的工作目录：PDFSPINE_WORK_DIR，默认 batch_qa 工作目录名后加 _pdfspine（同级目录）
WORK_DIR = under_root(_env("PDFSPINE_WORK_DIR", str(TEXT_WORK_DIR.with_name(TEXT_WORK_DIR.name + "_pdfspine"))))
print(f"工作目录 : {WORK_DIR}   [来源：{_source('PDFSPINE_WORK_DIR')}]   [{'已存在' if WORK_DIR.is_dir() else '尚未创建（首次写入时自动创建）'}]")
print(f"对比目录 : {TEXT_WORK_DIR}（batch_qa 文字层工作目录，只读）")
if WORK_DIR.resolve() == TEXT_WORK_DIR.resolve():
    raise ValueError(f"PDFSPINE_WORK_DIR 与 batch_qa 的工作目录相同（{WORK_DIR}），会覆盖文字层产物；请换一个目录")
if WORK_DIR.resolve().is_relative_to(PDF_DIR.resolve()):
    raise ValueError(f"工作目录落在 PDF 目录之内；PDF 目录按只读处理，请修改 PDFSPINE_WORK_DIR 或 PDF_DIR。WORK_DIR={WORK_DIR}，PDF_DIR={PDF_DIR}")
# 持久目录：PDFSPINE_PERSIST_DIR，默认 batch_qa 的 PERSIST_DIR 名后加 _pdfspine；都没设 = 不同步
_text_persist = _env("PERSIST_DIR", None, Path)
TEXT_PERSIST_DIR: Path | None = under_root(_text_persist) if _text_persist else None  # batch_qa 的持久目录：只读
_persist = _env("PDFSPINE_PERSIST_DIR", None, Path)
PERSIST_DIR: Path | None = (under_root(_persist) if _persist else
                            TEXT_PERSIST_DIR.with_name(TEXT_PERSIST_DIR.name + "_pdfspine") if TEXT_PERSIST_DIR else None)
print(f"持久目录 : {PERSIST_DIR}   [来源：{_source('PDFSPINE_PERSIST_DIR') if _persist else '由 PERSIST_DIR 推出'}]"
      if PERSIST_DIR else "持久目录 : （未设置，不同步）")
if PERSIST_DIR is not None and TEXT_PERSIST_DIR is not None and PERSIST_DIR.resolve() == TEXT_PERSIST_DIR.resolve():
    raise ValueError(f"PDFSPINE_PERSIST_DIR 与 batch_qa 的 PERSIST_DIR 相同（{PERSIST_DIR}），会覆盖文字层产物；请换一个目录")
if PERSIST_DIR is not None and PERSIST_DIR.resolve().is_relative_to(PDF_DIR.resolve()):
    raise ValueError(f"持久目录落在 PDF 目录之内；PDF 目录按只读处理，请修改 PDFSPINE_PERSIST_DIR 或 PDF_DIR。PERSIST_DIR={PERSIST_DIR}，PDF_DIR={PDF_DIR}")
MD_DIR = WORK_DIR / "md"        # 转出的 Markdown
STORE = WORK_DIR / "store"      # 索引库
# 对比用的文字层 md：batch_qa 工作目录的 md/，没有时用它持久目录里的 md/（只读）；都没有就只统计 pdfspine
TEXT_MD_DIR: Path | None = next((d for d in (TEXT_WORK_DIR / "md", TEXT_PERSIST_DIR / "md" if TEXT_PERSIST_DIR else None)
                                 if d is not None and d.is_dir()), None)

# --- 建库 ---
INDEX_SUMMARY = _env("INDEX_SUMMARY", False, _bool)  # True：建库时用 LLM（SUPERINDEX_INDEX_MODEL）写节点摘要；False：不调 LLM
# 文档元数据（与 INDEX_SUMMARY 无关）：每份文档一次 LLM 调用，抽公司 / 地区 / 主报告期 / 报告类型 / 一句话描述，存进 doc.json
# 的 metadata.doc_meta（description 为空时也填上）。模型用 SUPERINDEX_INDEX_MODEL，没设用 SUPERINDEX_CHAT_MODEL；都没配或调用失败时
# 只用文件名规则（如 AIA_AR2024 → FY2024 年报）。已建好的库重跑时自动补齐缺的元数据，不重建树。
DOC_META = _env("DOC_META", True, _bool)
# 建库时（元数据 / 摘要）模型接口额度不足或被限流：等多久（秒）后重试同一文档、每个文档最多等几次；0 = 不等（直接退回）。
# 某个文档等满仍被限流 = 判断为额度耗尽：本次运行后续文档不再等待（直接按失败处理，下次运行重试）。
# 等满仍失败：元数据退回文件名规则、不记为已完成（下次运行重新抽取）；摘要失败该文档记"失败"、不入库（下次运行重建）。
INDEX_RATE_LIMIT_WAIT = _env("INDEX_RATE_LIMIT_WAIT", 60.0, float)
INDEX_RATE_LIMIT_MAX_WAITS = _env("INDEX_RATE_LIMIT_MAX_WAITS", 30, int)

print(f"MD_DIR   : {MD_DIR}")
print(f"STORE    : {STORE}")
print(f"文字层 md: {TEXT_MD_DIR or '（没有，步骤 3 只统计 pdfspine）'}")
print(f"PDF 密码 : 已配置 {len(PDF_PASSWORDS)} 个候选密码")  # 只打印个数，绝不打印密码本身

In [ ]:
# ───────────── 持久化目录同步（仅当配置了 PERSIST_DIR；否则下面的函数什么都不做）─────────────
PERSIST_ITEMS = ("md", "store", "skipped_pdfs.json")  # 开始时从 PERSIST_DIR 恢复的内容


def _copy_file(src: Path, dst: Path) -> bool:
    """复制一个文件；dst 已有同样大小、且修改时间不早于 src（容差 1s，FUSE 时间戳精度低）的视为未变、跳过。
    返回是否复制了。只复制内容、不复制权限（FUSE 路径上 copystat 常失败）；修改时间尽量设成与 src 一致，失败忽略。"""
    st = src.stat()
    try:
        dst_st = dst.stat()
        if dst_st.st_size == st.st_size and dst_st.st_mtime >= st.st_mtime - 1:
            return False
    except OSError:
        pass
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(src, dst)
    try:
        os.utime(dst, (st.st_atime, st.st_mtime))
    except OSError:
        pass
    return True


def _copy_files(src: Path, dst: Path) -> tuple[int, int]:
    """把 src（文件或目录）复制到 dst，只复制有变化的文件；返回 (复制的文件数, 总文件数)。"""
    if src.is_file():
        return int(_copy_file(src, dst)), 1
    files = [p for p in src.rglob("*") if p.is_file()]
    return sum(_copy_file(p, dst / p.relative_to(src)) for p in files), len(files)


def _stale_docs(persist: Path) -> list[str]:
    """PERSIST_DIR/store/docs 里本地 store 已不存在的文档目录名（重建库时旧 doc_id 被删）；只列出，不删除。"""
    local, remote = WORK_DIR / "store" / "docs", persist / "store" / "docs"
    if not local.is_dir() or not remote.is_dir():
        return []
    return sorted(d.name for d in remote.iterdir() if d.is_dir() and not (local / d.name).exists())


def _persist_dir() -> Path | None:
    if PERSIST_DIR is None:
        return None
    if PERSIST_DIR.resolve() == WORK_DIR.resolve():
        print("PERSIST_DIR 与 WORK_DIR 相同，无需同步")
        return None
    return PERSIST_DIR


def restore_from_persist() -> None:
    """WORK_DIR 里还没有的 md/store/runs/skipped_pdfs.json，从 PERSIST_DIR 复制进来（集群重启后接着上次的进度）。"""
    persist = _persist_dir()
    if persist is None:
        return
    for name in PERSIST_ITEMS:
        src, dst = persist / name, WORK_DIR / name
        if not src.exists() or dst.exists():
            continue
        try:
            print(f"从 PERSIST_DIR 恢复 {name}（{_copy_files(src, dst)[1]} 个文件）")
        except Exception as exc:  # noqa: BLE001 - 恢复失败只警告：删掉复制了一半的残留，该项之后重新生成
            shutil.rmtree(dst, ignore_errors=True)
            print(f"警告：从 PERSIST_DIR 恢复 {name} 失败（{type(exc).__name__}: {exc}），将重新生成")


def sync_to_persist(label: str) -> None:
    """把 WORK_DIR 的内容复制到 PERSIST_DIR（只复制有变化的文件）；失败只警告、不中断。"""
    persist = _persist_dir()
    if persist is None:
        return
    try:
        copied = [f"{src.name}（复制 {n}/{total} 个有变化的文件）"
                  for src in sorted(WORK_DIR.iterdir()) for n, total in [_copy_files(src, persist / src.name)]
                  ] if WORK_DIR.is_dir() else []
        stale = _stale_docs(persist)
        print(f"[{label}] 已同步到 {persist}：" + ("、".join(copied) or "（WORK_DIR 里还没有内容）")
              + ("\n" if stale else ""))
        if stale:
            print(f"提示：PERSIST_DIR/store/docs 里有 {len(stale)} 个本地已不存在的旧文档目录（{'、'.join(stale[:5])}"
                  f"{'…' if len(stale) > 5 else ''}），未删除；集群重启恢复后可能出现同名文档的新旧两份。"
                  "如确认 PERSIST_DIR 不被其他工作目录共用，可手动删除这些目录。")
    except Exception as exc:  # noqa: BLE001
        print(f"警告：同步到 PERSIST_DIR 失败（{type(exc).__name__}: {exc}）；本地结果仍在 {WORK_DIR}")


restore_from_persist()

In [ ]:
# ───────────── 可选：公司自封装 Python SDK 适配器（默认关闭）─────────────
# 公司模型不是 OpenAI 协议、只能通过公司 Python SDK 调用时，把 USE_CUSTOM_LLM 改 True，
# 在 company_chat 里调用公司 SDK，并把模型名配成 company/<任意名>（上面"模型配置"的方式四；
# 问答用 SUPERINDEX_CHAT_MODEL，INDEX_SUMMARY=True 时建库摘要用 SUPERINDEX_INDEX_MODEL，同样写 company/...）。
# 模板把 company_chat 的返回值转成 litellm 需要的格式；流式是"假流式"（整段结果一次性吐出）。
# USE_CUSTOM_LLM=False 时本 cell 什么都不做（不 import litellm）。
USE_CUSTOM_LLM = _env("USE_CUSTOM_LLM", False, _bool)


def company_chat(messages: list[dict], tools: list[dict] | None) -> dict:
    """★ 在这里调用公司 SDK ★

    入参：
      messages  OpenAI 风格的消息列表：{"role": "system"/"user"/"assistant"/"tool", "content": str, ...}；
                assistant 消息可能带 "tool_calls"，tool 消息带 "tool_call_id"（图片等非文字内容已被去掉）。
      tools     OpenAI 风格的工具定义列表 [{"type": "function", "function": {"name", "description", "parameters"}}]，
                可能为 None。问答 agent 靠它调用 search_pages / get_page_content 等，模型必须能返回工具调用。
    返回：{"text": str | None, "tool_calls": [{"id": str, "name": str, "arguments": str(JSON)}]}
      要调用工具时 tool_calls 非空（text 可为 None）；给最终回答时 tool_calls 为空、text 是回答。
    """
    raise NotImplementedError("请在 company_chat 里调用公司 SDK")


def register_company_llm() -> None:
    """把 company_chat 注册成 litellm 的自定义 provider "company"。"""
    import litellm
    from litellm import CustomLLM
    from litellm.types.utils import GenericStreamingChunk

    def text_of(content: Any) -> str:
        if isinstance(content, list):  # 多段内容：只保留文字
            return "".join(p.get("text", "") for p in content if isinstance(p, dict) and p.get("type") == "text")
        return content if isinstance(content, str) else "" if content is None else json.dumps(content, ensure_ascii=False)

    def call(messages: list[dict], optional_params: dict) -> tuple[str | None, list[dict]]:
        result = company_chat([{**m, "content": text_of(m.get("content"))} for m in messages],
                              optional_params.get("tools"))
        calls = [{"id": c["id"], "type": "function", "function": {"name": c["name"], "arguments": c["arguments"]}}
                 for c in result.get("tool_calls") or []]
        return result.get("text"), calls

    def to_response(result: tuple[str | None, list[dict]], model_response: Any, model: str) -> Any:
        text, calls = result
        model_response.choices[0].message = litellm.Message(content=text, role="assistant", tool_calls=calls or None)
        model_response.choices[0].finish_reason = "tool_calls" if calls else "stop"
        model_response.model = model
        return model_response

    def to_chunks(result: tuple[str | None, list[dict]]) -> list[GenericStreamingChunk]:
        text, calls = result
        usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}  # SDK 有用量数据可在此填入
        chunks = [GenericStreamingChunk(text=text or "", is_finished=False, finish_reason="", usage=usage,
                                        index=0, tool_use=None)]
        chunks += [GenericStreamingChunk(text="", is_finished=False, finish_reason="", usage=usage, index=0,
                                         tool_use={**call, "index": i}) for i, call in enumerate(calls)]
        chunks[-1]["is_finished"] = True
        chunks[-1]["finish_reason"] = "tool_calls" if calls else "stop"
        return chunks

    class CompanyLLM(CustomLLM):
        def completion(self, model, messages, api_base, custom_prompt_dict, model_response, print_verbose,
                       encoding, api_key, logging_obj, optional_params, **kwargs):
            return to_response(call(messages, optional_params), model_response, model)

        async def acompletion(self, model, messages, api_base, custom_prompt_dict, model_response, print_verbose,
                              encoding, api_key, logging_obj, optional_params, **kwargs):
            return to_response(call(messages, optional_params), model_response, model)

        def streaming(self, model, messages, api_base, custom_prompt_dict, model_response, print_verbose,
                      encoding, api_key, logging_obj, optional_params, **kwargs):
            yield from to_chunks(call(messages, optional_params))

        async def astreaming(self, model, messages, api_base, custom_prompt_dict, model_response, print_verbose,
                             encoding, api_key, logging_obj, optional_params, **kwargs):
            for chunk in to_chunks(call(messages, optional_params)):
                yield chunk

    others = [p for p in (litellm.custom_provider_map or []) if p.get("provider") != "company"]
    litellm.custom_provider_map = [*others, {"provider": "company", "custom_handler": CompanyLLM()}]


if USE_CUSTOM_LLM:
    register_company_llm()
    print("已注册自定义 LLM provider：模型名写 company/<任意名>")

## 步骤 1：PDF → Markdown（pdfspine）

逐个 PDF 转换，`md/` 下已有同名 `.md` 就跳过（`FORCE_EXTRACT=True` 强制重转）。每个 `.md` 旁写一个 `.meta.json`，字段同 batch_qa（`extractor: "pdfspine"`，另记 `tables` 表格数）。

- 每页前 `<!-- page: N -->`，页间空一行（与 batch_qa 文字层同一格式，`superindex.md_ingest.PAGE_MARKER_RE` 可解析）；空页保留标记，页码与 PDF 一致。
- 表格（有框线的）转成 Markdown 表格；标题按字号在整份文档上统一分级（`#` / `##` / `###`）。
- 需要密码的 PDF：先试空密码，不行再按候选密码顺序试；都不对的跳过并标记 `skipped_wrong_password`（下次运行重新尝试）。所有页都没有文字的（扫描件）标记 `skipped_no_text_layer`。
- 单个 PDF 出问题不影响其它文档；汇总表 `status` 含义同 batch_qa。

In [ ]:
def list_pdfs(pdf_dir: Path) -> list[Path]:
    if not pdf_dir.is_dir():
        raise FileNotFoundError(
            f"PDF 目录不存在：{pdf_dir.resolve()}（来源：{_source('PDF_DIR')}；ROOT_DIR={ROOT_DIR}）。"
            f"修法：在 .env 里写 `PDF_DIR=<PDF 目录的绝对路径>`（.env 放在 {ROOT_DIR} 或 {Path.cwd().resolve()}），"
            "或设置同名环境变量；旧名 SI_NB_PDF_DIR 已不再生效")
    pdfs = sorted(p for p in pdf_dir.rglob("*") if p.is_file() and p.suffix.lower() == ".pdf")
    if not pdfs:
        raise FileNotFoundError(f"{pdf_dir.resolve()} 下没有 PDF 文件")
    return pdfs


SKIPPED = "skipped_no_text_layer"      # meta.json 里的 status：无文字层，已跳过
WRONG_PASSWORD = "skipped_wrong_password"  # meta.json 里的 status：需要密码但没配 / 候选密码都不对，已跳过
SKIP_LABELS = {SKIPPED: "无文字层", WRONG_PASSWORD: "密码不正确或未配置"}  # 被跳过的 PDF：状态 -> 原因（读题、汇总共用）
SKIP_STATUSES = tuple(SKIP_LABELS)


def extract_pdfspine(pdf: Path, md_path: Path) -> dict[str, Any]:
    """一个 PDF 用 pdfspine 离线转 Markdown（带页码标记、表格、# 标题），写 .md 与 .meta.json；返回汇总表的一行。
    需要密码的 PDF 由 pdfspine 用 PDF_PASSWORDS 在内存里解密，不落解密副本；跳过规则与 extract_text_layer 相同。"""
    from superindex.extractors.pdfspine import convert_pdf, markdown_stats

    res = convert_pdf(pdf, PDF_PASSWORDS)
    if res.status == WRONG_PASSWORD:
        reason = ("PDF 需要密码才能打开，但没有配置 PDF_INGEST_PASSWORD / PDF_PASSWORDS" if not PDF_PASSWORDS
                  else f"PDF 需要密码才能打开，已尝试的 {len(PDF_PASSWORDS)} 个候选密码均不正确")
        write_meta(pdf, status=WRONG_PASSWORD, encrypted=True, password_used=False, pages=None, reason=reason)
        return {"status": WRONG_PASSWORD, "pages": None, "headings": 0,
                "note": f"警告：{reason}，已跳过、不建库；改好 PDF_INGEST_PASSWORD / PDF_PASSWORDS 后重跑会自动重试"}
    if res.status == SKIPPED:
        write_meta(pdf, status=SKIPPED, encrypted=res.encrypted, password_used=res.password_used, pages=res.pages,
                   empty_pages=res.empty_pages, reason="PDF 没有文字层（扫描件/纯图片），pdfspine 抽取结果全为空；如需处理请改用 azure-di")
        return {"status": SKIPPED, "pages": res.pages, "headings": 0,
                "note": "警告：无文字层（扫描件？），已跳过、不建库；请改用 PDF_EXTRACTOR=\"azure-di\" 处理"}
    stats = markdown_stats(res.markdown)
    md_path.parent.mkdir(parents=True, exist_ok=True)
    md_path.write_text(res.markdown, encoding="utf-8")
    write_meta(pdf, status="ok", encrypted=res.encrypted, password_used=res.password_used, pages=res.pages,
               empty_pages=res.empty_pages, add_headings=False, headings_added=bool(stats["headings"]),
               heading_count=stats["headings"], tables=stats["tables"])
    note = f"警告：{len(res.empty_pages)} 页无文字 {res.empty_pages}（可能是整页图片）" if res.empty_pages else ""
    return {"status": "converted", "pages": res.pages, "headings": stats["headings"], "note": note}


def meta_path_of(pdf: Path) -> Path:
    return (MD_DIR / pdf.relative_to(PDF_DIR)).with_suffix(".meta.json")


def write_meta(pdf: Path, **fields: Any) -> None:
    meta = {"source": str(pdf.resolve()), "source_path": pdf.relative_to(PDF_DIR).as_posix(),
            "extractor": PDF_EXTRACTOR, **fields,
            "extracted_at": datetime.now().isoformat(timespec="seconds")}
    path = meta_path_of(pdf)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")


def read_meta(pdf: Path) -> dict[str, Any]:
    try:
        return json.loads(meta_path_of(pdf).read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return {}


print(f"MD_DIR {MD_DIR} 下已有 {sum(1 for _ in MD_DIR.rglob('*.md')) if MD_DIR.is_dir() else 0} 个 .md", flush=True)
pdfs = list_pdfs(PDF_DIR)
# 库里按文档名区分文档：不同文件夹的同名 PDF 建库时自动在文档名前加文件夹前缀区分（步骤 2），先醒目提示
_name_clashes = same_name_sources([p.relative_to(PDF_DIR).as_posix() for p in pdfs])
if _name_clashes:
    _prefixed = clash_names([p.relative_to(PDF_DIR).as_posix() for p in pdfs])
    print("!" * 80 + f"\n注意：{len(_name_clashes)} 组不同文件夹的同名 PDF，建库时已自动加文件夹前缀区分"
          "（只改这些文档的库内名称，其它文档名称不变）：", flush=True)
    for _name, _paths in _name_clashes.items():
        print(f"  {_name}: " + " | ".join(f"{_p} → {_prefixed[_p]}" for _p in _paths), flush=True)
    print("!" * 80, flush=True)
rows: list[dict[str, Any]] = []
step_started = time.perf_counter()
print(f"步骤 1：共 {len(pdfs)} 个 PDF，抽取方式 {PDF_EXTRACTOR}（pdfspine {pdfspine.__version__}）", flush=True)


def progress(i: int, pdf: Path, row: dict[str, Any], seconds: float) -> None:
    """实时打印一个 PDF 的处理结果（开始处理时已先打印了 "[i/n] 文件名 ..."，这里接着写在同一行）。"""
    pages = f"，{row['pages']} 页" if row.get("pages") else ""
    heads = f"，标题 {row['headings']}" if row.get("headings") else ""
    print(f" {row['status']}{pages}{heads}（{seconds:.1f}s，累计 {time.perf_counter() - step_started:.0f}s）", flush=True)


for i, pdf in enumerate(pdfs, start=1):
    print(f"[{i}/{len(pdfs)}] {pdf.relative_to(PDF_DIR)} ...", end="", flush=True)
    started = time.perf_counter()
    md_path = (MD_DIR / pdf.relative_to(PDF_DIR)).with_suffix(".md")
    meta = read_meta(pdf)
    # 已标记 skipped_wrong_password 的不在这里跳过：每次运行都会重新尝试打开（密码配置可能已改对）
    if not FORCE_EXTRACT and (md_path.exists() or meta.get("status") == SKIPPED):
        status = SKIPPED if meta.get("status") == SKIPPED and not md_path.exists() else "skipped_existing"
        note = "此前已标记为无文字层，不重复抽取；设 FORCE_EXTRACT=True 重试" if status == SKIPPED else ""
        if meta.get("empty_pages") and status != SKIPPED:
            note = f"警告：{len(meta['empty_pages'])} 页无文字 {meta['empty_pages']}"
        rows.append({"pdf": pdf.name, "status": status, "pages": meta.get("pages"),
                     "headings": meta.get("heading_count"), "note": note})
        progress(i, pdf, rows[-1], time.perf_counter() - started)
        continue
    if FORCE_EXTRACT and md_path.exists():
        md_path.unlink()  # 重转：避免上一次的 .md 在新结果为"无文字层"时残留
    try:
        result = extract_pdfspine(pdf, md_path)
    except Exception as exc:  # noqa: BLE001 - 单个 PDF 失败不中断其它文档
        result = {"status": "failed", "pages": None, "headings": None, "note": f"{type(exc).__name__}: {exc}"}
    rows.append({"pdf": pdf.name, **result})
    progress(i, pdf, rows[-1], time.perf_counter() - started)

# 汇总所有被跳过的 PDF（含此前运行标记的），写到 WORK_DIR/skipped_pdfs.json
skipped_pdfs = [{"pdf": str(p.relative_to(PDF_DIR)), "source": str(p.resolve()), "status": read_meta(p)["status"],
                 "reason": read_meta(p).get("reason"), "pages": read_meta(p).get("pages"),
                 "marked_at": read_meta(p).get("extracted_at")}
                for p in pdfs if read_meta(p).get("status") in SKIP_STATUSES]
WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / "skipped_pdfs.json").write_text(json.dumps(skipped_pdfs, ensure_ascii=False, indent=2), encoding="utf-8")

_counts: dict[str, int] = {}
for row in rows:
    _counts[row["status"]] = _counts.get(row["status"], 0) + 1
print(f"步骤 1 完成：{len(rows)} 个 PDF，用时 {time.perf_counter() - step_started:.0f}s；"
      + "，".join(f"{k} {v}" for k, v in sorted(_counts.items())), flush=True)
display(pd.DataFrame(rows))
for row in rows:
    if row["status"] in (*SKIP_STATUSES, "failed") or "警告" in row["note"]:
        print(f"[{row['status']}] {row['pdf']}: {row['note']}")
print(f"被跳过的 PDF：{len(skipped_pdfs)} 个，清单见 {WORK_DIR / 'skipped_pdfs.json'}")

## 步骤 2：建库（与 batch_qa 步骤 2 同一段代码）

对 `md/` 下每个 Markdown 调 `index_markdown`（`pdf=` 记录源 PDF 关联）。内容没变会自动跳过；
`INDEX_SUMMARY=False` 时不调 LLM。

每个文档记下来源 PDF 相对 `PDF_DIR` 的路径（`doc_meta.source_path`；旧 md 从 `.meta.json` 或按文件名回 `PDF_DIR` 查找），
报告类型按「来源文件夹 > 文件名缩写/全称 > LLM」定（映射见 `config/routing_policy.yaml` 的 `report_types`），
报告期按文件名规范化（`FY2022` / `1H2022` / `2022Q3` / `2022M03` …，文件名能解析就以文件名为准）。
已建库的文档只按规则补全这些字段（不调 LLM、不重建树）；最后打印一行统计：各报告类型文档数、有期间的文档数、来源文件夹无法识别类型的文档数。

In [ ]:
settings = LLMSettings.resolve()
summary_model = None
if INDEX_SUMMARY:
    summary_model = settings.require("index")  # 没配 SUPERINDEX_INDEX_MODEL 会给出明确提示
    configure_litellm()
doc_meta_model = (settings.index_model or settings.chat_model) if DOC_META else None
if doc_meta_model:
    configure_litellm()

md_files = sorted(MD_DIR.rglob("*.md")) if MD_DIR.is_dir() else []
if not md_files:
    raise RuntimeError(f"{MD_DIR} 下没有 Markdown，请先成功运行步骤 1")
pdf_by_stem = page_images.pdf_index(PDF_DIR)
# 不同文件夹的同名 PDF：建库前基于 PDF_DIR 下全部 PDF 一次算好库内文档名（冲突的加文件夹前缀，其余不变），每次运行结果一致；
# 并删掉加前缀之前库里留下的那份无前缀同名文档
_pdf_root = PDF_DIR.resolve()
_all_sources = [p.relative_to(_pdf_root).as_posix() for paths in pdf_by_stem.values() for p in paths]
DOC_NAMES = clash_names(_all_sources)
for _gone in drop_clash_leftovers(STORE, _all_sources):
    print(f"已删除库里加前缀前的无前缀同名文档：{_gone}", flush=True)

index_rows: list[dict[str, Any]] = []
# index_markdown 内部会 asyncio.run(...)，而 Notebook 的主线程已有运行中的事件循环，直接调用会报
# "asyncio.run() cannot be called from a running event loop"；放到单独的工作线程里执行（该线程没有事件循环）。
index_pool = ThreadPoolExecutor(max_workers=1)


def pdf_to_link(md_path: Path) -> Path | None:
    """index_markdown(pdf=...) 的源 PDF 关联，只为页面截图服务（默认关闭）。
    步骤 1 用密码打开的 PDF 不关联：页面截图用的 pypdfium2 没有密码就打不开，传 pdf= 只会得到一条误导性警告
    （"PDF ... not linked"）——即需要密码的 PDF 不支持页面截图，文字问答不受影响。"""
    try:
        if json.loads(md_path.with_suffix(".meta.json").read_text(encoding="utf-8")).get("password_used"):
            return None
    except (OSError, ValueError):
        pass
    return page_images.find_pdf(md_path, pdf_by_stem)


rate_limit = RateLimitPolicy(INDEX_RATE_LIMIT_WAIT, INDEX_RATE_LIMIT_MAX_WAITS)  # 额度耗尽后，后续文档不再等待
index_started = time.perf_counter()
print(f"步骤 2：共 {len(md_files)} 个文档，摘要 {'开（调用模型）' if INDEX_SUMMARY else '关'}", flush=True)
for i, md_path in enumerate(md_files, start=1):
    print(f"[{i}/{len(md_files)}] {md_path.relative_to(MD_DIR)} ...", end="", flush=True)
    started = time.perf_counter()
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            source = pdf_source_path(md_path, PDF_DIR, MD_DIR, pdf_by_stem)
        res = index_pool.submit(index_markdown, md_path, STORE, summary_model=summary_model,
                                    backend=settings.index_backend(), force=False,
                                    pdf=pdf_to_link(md_path), doc_meta=DOC_META,
                                    doc_meta_model=doc_meta_model, rate_limit=rate_limit,
                                    source_path=source, name=DOC_NAMES.get(source or "")).result()
        doc_meta = res.doc_meta or {}
        period = " ".join(str(doc_meta[k]) for k in ("period", "report_type") if doc_meta.get(k))
        if not res.skipped:
            status = "已建库"
        elif res.doc_meta_added:
            status = "跳过（未变），补元数据 " + (f"{period}（{doc_meta.get('source')}）" if doc_meta else "")
        else:
            status = "跳过（未变）"
        if doc_meta.get("llm_failed"):
            status += "；元数据 LLM 调用失败，已退回文件名规则（下次运行重试）"
        index_rows.append({"doc_id": res.doc_id, "文档": res.name, "页数": res.pages, "节点数": res.nodes,
                           "状态": status,
                           "报告期": " ".join(str(doc_meta[k]) for k in ("period", "report_type") if doc_meta.get(k))
                           + (f"（{doc_meta.get('source')}）" if doc_meta else ""),
                           "页码标记": res.has_markers, "警告": "；".join(res.warnings)})
    except Exception as exc:  # noqa: BLE001 - 单个文档失败不中断其它文档
        index_rows.append({"doc_id": None, "文档": md_path.name, "页数": None, "节点数": None,
                           "状态": f"失败：{type(exc).__name__}: {exc}", "报告期": "", "页码标记": None, "警告": ""})
    print(f" {index_rows[-1]['状态'][:80]}（{time.perf_counter() - started:.1f}s，"
          f"累计 {time.perf_counter() - index_started:.0f}s）", flush=True)
index_pool.shutdown()
if DOC_META:  # 库里其它缺元数据 / 来源路径、或规则已更新的文档（如 md 已不在 MD_DIR 下）也补上；不重建树
    _filled = backfill_doc_meta(STORE, model=doc_meta_model, backend=settings.index_backend(),
                                rate_limit=rate_limit, pdf_dir=PDF_DIR)
    if _filled:
        print(f"文档元数据补齐 {len(_filled)} 个文档", flush=True)
_with_meta = [m for m in DocStore(str(STORE)).list_metas() if (m.get("metadata") or {}).get("doc_meta")]
print(f"文档元数据：{'开' if DOC_META else '关'}（模型 {doc_meta_model or '无，仅文件名规则'}）；"
      f"有元数据 {len(_with_meta)} 个，其中有报告期 "
      f"{sum(1 for m in _with_meta if m['metadata']['doc_meta'].get('period'))} 个", flush=True)
print("文档元数据统计：" + doc_meta_stats(STORE), flush=True)
print(f"步骤 2 完成：{len(index_rows)} 个文档，用时 {time.perf_counter() - index_started:.0f}s", flush=True)
display(pd.DataFrame(index_rows))
sync_to_persist("建库后")

## 步骤 3：对比统计（pdfspine vs 文字层）

只统计数字，不输出任何文档内容。按 `PDF_DIR` 下的文件夹分组，每组两行（`pdfspine`；有文字层 `md/` 时再加 `text-layer`）：

- `PDF 数`、`成功`（有 `.md`）、`跳过`（`meta.json` 标记无文字层 / 密码不正确）、`失败/未转换`（两者都没有）；
- 成功文档的 `页数`、`空页数`、`表格数`（Markdown 表格的个数）、`表格行数`（含表头行）、`标题数`（`#` 标题行）、`平均每页字符数`（页内文字去首尾空白后的字符数 ÷ 页数）。

文字层的 `表格数` 通常为 0（PyPDF2 只给纯文本），`标题数` 来自 flash 补标题（`ADD_HEADINGS`）。结果另存 `WORK_DIR/compare.csv`。

In [ ]:
def folder_of(pdf: Path) -> str:
    folder = pdf.relative_to(PDF_DIR).parent.as_posix()
    return "（根目录）" if folder == "." else folder


def doc_stats(md_dir: Path, pdf: Path) -> dict[str, Any]:
    """一个 PDF 在 md_dir 下的状态与统计：成功（有 .md）/ 跳过（meta.json 标记）/ 失败或未转换。"""
    stem = md_dir / pdf.relative_to(PDF_DIR)
    md_file, meta_file = stem.with_suffix(".md"), stem.with_suffix(".meta.json")
    if md_file.is_file():
        return {"state": "成功", **markdown_stats(md_file.read_text(encoding="utf-8", errors="replace"))}
    try:
        status = json.loads(meta_file.read_text(encoding="utf-8")).get("status")
    except (OSError, ValueError):
        status = None
    return {"state": "跳过" if status in SKIP_STATUSES else "失败/未转换"}


def compare_rows(label: str, md_dir: Path) -> list[dict[str, Any]]:
    per_folder: dict[str, list[dict[str, Any]]] = {}
    for pdf in pdfs:
        per_folder.setdefault(folder_of(pdf), []).append(doc_stats(md_dir, pdf))
    per_folder["合计"] = [s for group in list(per_folder.values()) for s in group]
    out = []
    for folder, stats in per_folder.items():
        ok = [s for s in stats if s["state"] == "成功"]
        pages = sum(s["pages"] for s in ok)
        out.append({"文件夹": folder, "抽取方式": label, "PDF 数": len(stats), "成功": len(ok),
                    "跳过": sum(1 for s in stats if s["state"] == "跳过"),
                    "失败/未转换": sum(1 for s in stats if s["state"] == "失败/未转换"),
                    "页数": pages, "空页数": sum(s["empty_pages"] for s in ok),
                    "表格数": sum(s["tables"] for s in ok), "表格行数": sum(s["table_rows"] for s in ok),
                    "标题数": sum(s["headings"] for s in ok),
                    "平均每页字符数": round(sum(s["chars"] for s in ok) / pages) if pages else None})
    return out


compare = compare_rows(PDF_EXTRACTOR, MD_DIR)
if TEXT_MD_DIR is not None:
    _text = compare_rows("text-layer", TEXT_MD_DIR)
    compare = [row for pair in zip(compare, _text) for row in pair]  # 同一文件夹的两行挨着
compare_df = pd.DataFrame(compare)
compare_df.to_csv(WORK_DIR / "compare.csv", index=False, encoding="utf-8-sig")
print(f"文字层 md：{TEXT_MD_DIR or '没有（只统计 pdfspine）'}；对比表已存 {WORK_DIR / 'compare.csv'}")
display(compare_df)

## 用 batch_qa 跑问答（只改配置，不改代码）

`batch_qa.ipynb` 支持 `PDF_EXTRACTOR="pdfspine"`（调用同一个 `superindex.extractors.pdfspine.convert_pdf`），工作目录布局与本 Notebook 相同。运行 batch_qa 时设：

- `NB_WORK_DIR` = 本 Notebook 的 `WORK_DIR`（下一格会打印）；
- `PDF_EXTRACTOR=pdfspine`；
- `PERSIST_DIR` = 本 Notebook 的持久目录（如有）；
- Databricks 上 batch_qa 的「环境安装」cell 命令里加 `pdfspine==0.12.0`。

设法：本地在 `nbconvert` 命令前加环境变量（如 `NB_WORK_DIR=... PDF_EXTRACTOR=pdfspine uv run --group notebook --extra pdfspine jupyter nbconvert ...`，进程环境变量优先于 `.env`）；
Databricks 上把这几项写进 batch_qa「模型配置」cell 的 `LLM_CONFIG`（该 cell 把字典里每一项写成环境变量，早于 `.env` 生效），如 `"NB_WORK_DIR": "/local_disk0/..."`、`"PDF_EXTRACTOR": "pdfspine"`。

这样 batch_qa 的步骤 1 看到已有 `.md`（`extractor` 一致）直接跳过、不会重新转换；本 Notebook 跳过 / 失败的 PDF 会由 batch_qa 用 pdfspine 重试，不会混入文字层结果；
步骤 2 内容没变直接跳过；问答结果写在该工作目录的 `runs/` 下，与文字层的运行结果分开。

**不要**用 `PDF_EXTRACTOR=text-layer` 指向这份目录：已有的 `.md` 虽然不会被覆盖（只提示抽取方式不同），但没有 `.md` 的 PDF 会按文字层转换，两种结果混在一个库里。

In [ ]:
sync_to_persist("完成")
print("用 batch_qa.ipynb 对这份库问答时设置：")
print(f"  NB_WORK_DIR={WORK_DIR}")
print("  PDF_EXTRACTOR=pdfspine")
if PERSIST_DIR is not None:
    print(f"  PERSIST_DIR={PERSIST_DIR}")